In [1]:
import os

os.chdir(r"C:\Users\HP USER\OneDrive\Desktop\transfer-learning-kaduna-house-price")

print("Current folder:", os.getcwd())
print("Files in data folder:", os.listdir("data"))

Current folder: C:\Users\HP USER\OneDrive\Desktop\transfer-learning-kaduna-house-price
Files in data folder: ['data_kaggle.csv', 'Strict_Kaduna_South_Dataset.csv']


In [2]:
import pandas as pd
import numpy as np

Load Malaysia dataset only

In [3]:
import pandas as pd

malaysia_df = pd.read_csv("data/data_kaggle.csv")

print("Original shape:", malaysia_df.shape)
print(malaysia_df.columns.tolist())

Original shape: (53883, 8)
['Location', 'Price', 'Rooms', 'Bathrooms', 'Car Parks', 'Property Type', 'Size', 'Furnishing']


Select only relevant columns

In [4]:
selected_columns = [
    "Rooms",
    "Size",
    "Bathrooms",
    "Furnishing",
    "Property Type",
    "Price"
]

malaysia_df = malaysia_df[selected_columns]

print("After selecting columns:", malaysia_df.shape)
malaysia_df.head()

After selecting columns: (53883, 6)


,Rooms,Size,Bathrooms,Furnishing,Property Type,Price
0,2+1,"Built-up : 1,335 sq. ft.",3.0,Fully Furnished,Serviced Residence,"RM 1,250,000"
1,6,Land area : 6900 sq. ft.,7.0,Partly Furnished,Bungalow,"RM 6,800,000"
2,3,"Built-up : 1,875 sq. ft.",4.0,Partly Furnished,Condominium (Corner),"RM 1,030,000"
3,NaN,NaN,NaN,NaN,NaN,NaN
4,4+1,"Built-up : 1,513 sq. ft.",3.0,Partly Furnished,Condominium (Corner),"RM 900,000"


# Check missing values before cleaning

In [5]:
# Check missing values before cleaning
print("Shape before missing value handling:", malaysia_df.shape)
print(malaysia_df.isnull().sum())

Shape before missing value handling: (53883, 6)
Rooms            1706
Size             1063
Bathrooms        2013
Furnishing       6930
Property Type      25
Price             248
dtype: int64


# Drop rows with missing values, following the base paper approach

In [6]:
# Drop rows with missing values, following the base paper approach
malaysia_df = malaysia_df.dropna()

print("Shape after dropping missing values:", malaysia_df.shape)
print(malaysia_df.isnull().sum())

Shape after dropping missing values: (45207, 6)
Rooms            0
Size             0
Bathrooms        0
Furnishing       0
Property Type    0
Price            0
dtype: int64


In [7]:
# Check current data types before numeric cleaning
malaysia_df.dtypes

Rooms                str
Size                 str
Bathrooms        float64
Furnishing           str
Property Type        str
Price                str
dtype: object

In [8]:
# Clean Rooms column
malaysia_df["Rooms"] = (
    malaysia_df["Rooms"]
    .astype(str)
    .str.extract(r"(\d+\.?\d*)")
    .astype(float)
)

# Clean Bathrooms column
malaysia_df["Bathrooms"] = (
    malaysia_df["Bathrooms"]
    .astype(str)
    .str.extract(r"(\d+\.?\d*)")
    .astype(float)
)

# Clean Size column
malaysia_df["Size"] = (
    malaysia_df["Size"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+\.?\d*)")
    .astype(float)
)

# Clean Price column
malaysia_df["Price"] = (
    malaysia_df["Price"]
    .astype(str)
    .str.replace("RM", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.extract(r"(\d+\.?\d*)")
    .astype(float)
)

malaysia_df.head()

,Rooms,Size,Bathrooms,Furnishing,Property Type,Price
0,2.0,1335.0,3.0,Fully Furnished,Serviced Residence,1250000.0
1,6.0,6900.0,7.0,Partly Furnished,Bungalow,6800000.0
2,3.0,1875.0,4.0,Partly Furnished,Condominium (Corner),1030000.0
4,4.0,1513.0,3.0,Partly Furnished,Condominium (Corner),900000.0
5,4.0,7200.0,5.0,Partly Furnished,Bungalow,5350000.0


## 5. Checking for Failed Numeric Conversion

After converting the selected columns to numerical format, some values may fail to convert properly.  
Such failed values become missing values again, so they are removed before continuing.

In [9]:
# Check missing values after numeric conversion
malaysia_df.isnull().sum()

Rooms            737
Size              46
Bathrooms          0
Furnishing         0
Property Type      0
Price              0
dtype: int64

In [10]:
# Drop rows where numeric conversion failed
malaysia_df = malaysia_df.dropna()

print("Shape after numeric cleaning:", malaysia_df.shape)
malaysia_df.head()

Shape after numeric cleaning: (44425, 6)


,Rooms,Size,Bathrooms,Furnishing,Property Type,Price
0,2.0,1335.0,3.0,Fully Furnished,Serviced Residence,1250000.0
1,6.0,6900.0,7.0,Partly Furnished,Bungalow,6800000.0
2,3.0,1875.0,4.0,Partly Furnished,Condominium (Corner),1030000.0
4,4.0,1513.0,3.0,Partly Furnished,Condominium (Corner),900000.0
5,4.0,7200.0,5.0,Partly Furnished,Bungalow,5350000.0


In [11]:
original_cols = [
    "Rooms",
    "Size",
    "Bathrooms",
    "Furnishing",
    "Property Type",
    "Price"
]

print("Duplicates before conversion:",
      malaysia_df.duplicated(subset=original_cols).sum())

Duplicates before conversion: 6093


## Removing duplicates based on original Malaysia dataset features

In [12]:
original_cols = [
    "Rooms",
    "Size",
    "Bathrooms",
    "Furnishing",
    "Property Type",
    "Price"
]

malaysia_df = malaysia_df.drop_duplicates(subset=original_cols)

print("Shape after removing duplicates correctly:", malaysia_df.shape)

Shape after removing duplicates correctly: (38332, 6)


In [13]:
# Remove duplicate rows
malaysia_df = malaysia_df.drop_duplicates()

print("Shape after removing duplicates:", malaysia_df.shape)

Shape after removing duplicates: (38332, 6)


## 6. Converting Malaysia Price from Ringgit to Naira

The Malaysia dataset price is originally in Malaysian Ringgit (MYR).  
Since the Kaduna dataset is in Nigerian Naira, the Malaysia price is converted to Naira before alignment.

In [14]:
MYR_TO_USD = 0.25

malaysia_df["Price_USD"] = malaysia_df["Price"] * MYR_TO_USD

malaysia_df[["Price", "Price_USD"]].head()

,Price,Price_USD
0,1250000.0,312500.0
1,6800000.0,1700000.0
2,1030000.0,257500.0
4,900000.0,225000.0
5,5350000.0,1337500.0


In [15]:
malaysia_df.drop(columns=["Price"], inplace=True)

In [16]:
# Remove duplicate rows
malaysia_df = malaysia_df.drop_duplicates()

print("Shape after removing duplicates:", malaysia_df.shape)

Shape after removing duplicates: (38332, 6)


In [17]:
print(malaysia_df.columns)

Index(['Rooms', 'Size', 'Bathrooms', 'Furnishing', 'Property Type',
       'Price_USD'],
      dtype='str')


In [18]:
malaysia_df.shape

(38332, 6)

Use only one price column

In [19]:
print(malaysia_df.columns)

Index(['Rooms', 'Size', 'Bathrooms', 'Furnishing', 'Property Type',
       'Price_USD'],
      dtype='str')


Confirm no duplicates remain

In [20]:
original_cols = [
    "Rooms",
    "Size",
    "Bathrooms",
    "Furnishing",
    "Property Type",
    "Price_USD"
]

print("Remaining duplicates:",
      malaysia_df.duplicated(subset=original_cols).sum())

Remaining duplicates: 0


Confirm no missing values

In [21]:
print(malaysia_df.isnull().sum())

Rooms            0
Size             0
Bathrooms        0
Furnishing       0
Property Type    0
Price_USD        0
dtype: int64


Confirm correct data types

In [22]:
print(malaysia_df.dtypes)

Rooms            float64
Size             float64
Bathrooms        float64
Furnishing           str
Property Type        str
Price_USD        float64
dtype: object


Check for unrealistic values

In [23]:
malaysia_df.describe()

,Rooms,Size,Bathrooms,Price_USD
count,38332.000000,38332.000000,38332.000000,3.833200e+04
mean,3.312089,2231.160922,3.171319,4.673070e+05
std,1.331076,9398.381833,1.667441,2.132842e+06
min,1.000000,0.000000,1.000000,7.700000e+01
25%,3.000000,915.000000,2.000000,1.500000e+05
50%,3.000000,1290.000000,3.000000,2.625000e+05
75%,4.000000,2207.000000,4.000000,5.200000e+05
max,20.000000,820000.000000,20.000000,4.000000e+08


Check unique categories (avoid messy categories)

In [24]:
print(malaysia_df["Furnishing"].unique())
print(malaysia_df["Property Type"].unique())

<StringArray>
['Fully Furnished', 'Partly Furnished', 'Unfurnished', 'Unknown']
Length: 4, dtype: str
<StringArray>
[                       'Serviced Residence',
                                  'Bungalow',
                      'Condominium (Corner)',
                       'Semi-detached House',
         '2-sty Terrace/Link House (EndLot)',
                  'Apartment (Intermediate)',
   '2-sty Terrace/Link House (Intermediate)',
                   'Bungalow (Intermediate)',
        'Semi-detached House (Intermediate)',
                         'Bungalow (Corner)',
         'Serviced Residence (Intermediate)',
                               'Condominium',
                'Condominium (Intermediate)',
                      'Condominium (EndLot)',
               'Serviced Residence (Corner)',
   '3-sty Terrace/Link House (Intermediate)',
               'Serviced Residence (Duplex)',
                  '2-sty Terrace/Link House',
         '2-sty Terrace/Link House (Corner)',
 '2.5-sty 

## Removing Unknown Values from Furnishing

In [25]:
malaysia_df = malaysia_df[malaysia_df["Furnishing"] != "Unknown"]

print(malaysia_df["Furnishing"].unique())

<StringArray>
['Fully Furnished', 'Partly Furnished', 'Unfurnished']
Length: 3, dtype: str


In [26]:
malaysia_df['Furnishing'].unique()

<StringArray>
['Fully Furnished', 'Partly Furnished', 'Unfurnished']
Length: 3, dtype: str

feature engineering turning fully furnished to new, partly furnished to fair and unfurnished to old

In [27]:
def map_condition(furnishing):
    f = str(furnishing).strip().lower()
    
    if "fully" in f:
        return "New"
    elif "partly" in f or "partial" in f:
        return "Fair"
    elif "unfurnished" in f:
        return "Old"


In [28]:
malaysia_df["Condition"] = malaysia_df["Furnishing"].apply(map_condition)
malaysia_df[["Furnishing", "Condition"]].head()

,Furnishing,Condition
0,Fully Furnished,New
1,Partly Furnished,Fair
2,Partly Furnished,Fair
4,Partly Furnished,Fair
5,Partly Furnished,Fair


In [29]:
malaysia_df = malaysia_df.drop(columns=["Furnishing"])

In [30]:
print(malaysia_df.columns)

Index(['Rooms', 'Size', 'Bathrooms', 'Property Type', 'Price_USD',
       'Condition'],
      dtype='str')


## Simplifying Property Type Categories

In [31]:
print(malaysia_df["Property Type"].value_counts())

Property Type
Condominium                            7522
Condominium (Corner)                   4719
Serviced Residence                     4397
Condominium (Intermediate)             4144
Serviced Residence (Intermediate)      2485
                                       ... 
Flat (Penthouse)                          1
2.5-sty Terrace/Link House (Duplex)       1
Apartment (Triplex)                       1
Semi-detached House (SOHO)                1
4.5-sty Terrace/Link House (Corner)       1
Name: count, Length: 93, dtype: int64


In [32]:
print(malaysia_df["Property Type"].unique())

<StringArray>
[                       'Serviced Residence',
                                  'Bungalow',
                      'Condominium (Corner)',
                       'Semi-detached House',
         '2-sty Terrace/Link House (EndLot)',
                  'Apartment (Intermediate)',
   '2-sty Terrace/Link House (Intermediate)',
                   'Bungalow (Intermediate)',
        'Semi-detached House (Intermediate)',
                         'Bungalow (Corner)',
         'Serviced Residence (Intermediate)',
                               'Condominium',
                'Condominium (Intermediate)',
                      'Condominium (EndLot)',
               'Serviced Residence (Corner)',
   '3-sty Terrace/Link House (Intermediate)',
               'Serviced Residence (Duplex)',
                  '2-sty Terrace/Link House',
         '2-sty Terrace/Link House (Corner)',
 '2.5-sty Terrace/Link House (Intermediate)',
         '3-sty Terrace/Link House (Corner)',
         '3-sty Terr

Mapping Function

In [33]:
def map_property_type(value):
    value = str(value).lower().strip()

    # Self-contain: small studio-type units
    if "studio" in value:
        return "Self-contain"

    # Bungalow: detached bungalow or land-type properties
    elif "bungalow" in value or "land" in value:
        return "Bungalow"

    # Duplex: private multi-level / compound-style houses
    elif (
        "terrace" in value
        or "link house" in value
        or "townhouse" in value
        or "semi-detached" in value
        or "duplex" in value
        or "triplex" in value
        or "cluster house" in value
    ):
        return "Duplex"

    # Flat: shared multi-unit buildings
    elif (
        "condominium" in value
        or "apartment" in value
        or "flat" in value
        or "serviced residence" in value
        or "soho" in value
        or "penthouse" in value
    ):
        return "Flat"

    # Safe fallback
    else:
        return "Flat"



### 📌 Apply Mapping


In [34]:
malaysia_df["Property Type"] = malaysia_df["Property Type"].apply(map_property_type)



### 📌 Check Unique Values

Verify result

In [35]:
malaysia_df["Property Type"].unique()

<StringArray>
['Flat', 'Bungalow', 'Duplex', 'Self-contain']
Length: 4, dtype: str



### 📌 Check Distribution

In [36]:
malaysia_df["Property Type"].value_counts()

Property Type
Flat            27448
Duplex           7839
Bungalow         2648
Self-contain       19
Name: count, dtype: int64

In [37]:
malaysia_df.columns

Index(['Rooms', 'Size', 'Bathrooms', 'Property Type', 'Price_USD',
       'Condition'],
      dtype='str')

load Kaduna dataset

In [38]:
kaduna_df = pd.read_csv("data/Strict_Kaduna_South_Dataset.csv")

print("Kaduna original shape:", kaduna_df.shape)
print(kaduna_df.columns.tolist())

kaduna_df.head()

Kaduna original shape: (37, 11)
['Property_Type', 'Bedrooms', 'Bathrooms', 'Size_sqm', 'Location', 'LGA', 'Condition', 'Security', 'Flood_Risk', 'Proximity', 'Price']


,Property_Type,Bedrooms,Bathrooms,Size_sqm,Location,LGA,Condition,Security,Flood_Risk,Proximity,Price
0,Bungalow,3,2,140,Barnawa,Kaduna South,New,High,Low,Near,48000000
1,Bungalow,3,2,150,Barnawa,Kaduna South,New,High,Low,Near,52000000
2,Bungalow,3,2,135,Barnawa,Kaduna South,Fair,Medium,Low,Near,43000000
3,Bungalow,3,2,145,Barnawa,Kaduna South,New,High,Low,Near,50000000
4,Flat,2,2,100,Kakuri,Kaduna South,Fair,Medium,Medium,Near,25000000



---

# 📌 2. Check Basic Info (Data Types + Null Overview)

In [39]:
kaduna_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Property_Type  37 non-null     str  
 1   Bedrooms       37 non-null     int64
 2   Bathrooms      37 non-null     int64
 3   Size_sqm       37 non-null     int64
 4   Location       37 non-null     str  
 5   LGA            37 non-null     str  
 6   Condition      37 non-null     str  
 7   Security       37 non-null     str  
 8   Flood_Risk     37 non-null     str  
 9   Proximity      37 non-null     str  
 10  Price          37 non-null     int64
dtypes: int64(4), str(7)
memory usage: 3.3 KB



---

# 📌 3. Check Missing Values (Per Column)

In [40]:

kaduna_df.isnull().sum()

Property_Type    0
Bedrooms         0
Bathrooms        0
Size_sqm         0
Location         0
LGA              0
Condition        0
Security         0
Flood_Risk       0
Proximity        0
Price            0
dtype: int64


---

# 📌 4. Check Missing Values (Percentage)

In [41]:
(kaduna_df.isnull().sum() / len(kaduna_df)) * 100

Property_Type    0.0
Bedrooms         0.0
Bathrooms        0.0
Size_sqm         0.0
Location         0.0
LGA              0.0
Condition        0.0
Security         0.0
Flood_Risk       0.0
Proximity        0.0
Price            0.0
dtype: float64



# 📌 5. Inspect Unique Values (Important Before Cleaning)

### 🔹 For Categorical Columns

In [42]:
categorical_cols = kaduna_df.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    print(f"\n{col}")
    print(kaduna_df[col].unique())


Property_Type
<StringArray>
['Bungalow', 'Flat', 'Duplex', 'Self-contain']
Length: 4, dtype: str

Location
<StringArray>
[     'Barnawa',       'Kakuri',   'Tudun Wada',  'Kabala West',
     'Kinkinau', 'Kurmin Mashi',   'Television']
Length: 7, dtype: str

LGA
<StringArray>
['Kaduna South', 'kaduna south', 'Kaduna south']
Length: 3, dtype: str

Condition
<StringArray>
['New', 'Fair', 'Old', 'Good', 'Excellent', 'Poor']
Length: 6, dtype: str

Security
<StringArray>
['High', 'Medium', 'Low', 'low']
Length: 4, dtype: str

Flood_Risk
<StringArray>
['Low', 'Medium', 'High', 'low']
Length: 4, dtype: str

Proximity
<StringArray>
['Near', 'Moderate', 'Far']
Length: 3, dtype: str


C:\Users\HP USER\AppData\Local\Temp\ipykernel_19480\690665154.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = kaduna_df.select_dtypes(include=["object"]).columns



---

### 🔹 For Numerical Columns

In [43]:
numerical_cols = kaduna_df.select_dtypes(include=["int64", "float64"]).columns

for col in numerical_cols:
    print(f"\n{col}")
    print("Min:", kaduna_df[col].min())
    print("Max:", kaduna_df[col].max())


Bedrooms
Min: 1
Max: 7

Bathrooms
Min: 1
Max: 7

Size_sqm
Min: 35
Max: 800

Price
Min: 2500000
Max: 350000000


# 📌 6. Check Duplicate Rows

In [44]:
kaduna_df.duplicated().sum()

np.int64(0)

# 📌 7. Inspect Rows With Missing Values

In [45]:
kaduna_df[kaduna_df.isnull().any(axis=1)]

,Property_Type,Bedrooms,Bathrooms,Size_sqm,Location,LGA,Condition,Security,Flood_Risk,Proximity,Price



---

# 📌 8. Check Distribution of Key Features

### Property Type

In [46]:
kaduna_df["Property_Type"].value_counts()

Property_Type
Flat            18
Bungalow        10
Duplex           6
Self-contain     3
Name: count, dtype: int64

In [47]:
### Location
kaduna_df["Location"].value_counts()

Location
Barnawa         13
Kabala West      7
Kakuri           4
Tudun Wada       4
Kinkinau         3
Kurmin Mashi     3
Television       3
Name: count, dtype: int64

In [ ]:
### Price
kaduna_df["Price"].describe()

count    3.700000e+01
mean     4.616216e+07
std      6.200885e+07
min      2.500000e+06
25%      1.500000e+07
50%      2.600000e+07
75%      4.500000e+07
max      3.500000e+08
Name: Price, dtype: float64


---

# 🔥 Step 4: Drop Irrelevant Columns (for alignment)

These are NOT in Malaysia:

- LGA ❌  
- Security ❌  
- Flood_Risk ❌  
- Proximity ❌  

In [49]:
kaduna_df = kaduna_df.drop(columns=[
    "LGA", 
    "Security", 
    "Flood_Risk", 
    "Proximity"
])

```python
# Set exchange rate
NGN_TO_USD = 1500

# Convert Price_NGN to USD


In [50]:
# Set exchange rate
NGN_TO_USD = 1500

# Convert Price_NGN to USD
kaduna_df["Price_USD"] = kaduna_df["Price_NGN"] / NGN_TO_USD

KeyError: 'Price_NGN'